#📚 文系大学生による『実践データ分析100本ノック＆LLM活用20本ノック』学習ログ
**著： 岡野 菜月（四国大学文学部 / 28卒）**

**GitHub： https://github.com/NatsukiOkano**

##📝 このノートブックの概要
＊ 書籍「Python 実践データ分析100本ノック 第3版」（秀和システム新社刊）の学習過程を記録したものです。

＊ 最大の特徴は、「文学部の視点でIT・データ分析を再解釈する」という試みにあります。

＊ 非情報系の学習者が直面する「専門用語の壁」を取り払うため、すべての処理に対し、論理的かつ直感的な「完全独自解説」を記述しています。

##⚠️ 注意事項
- **著作権保護の観点から、データセットおよび実行結果は同梱しておりません。**

### 教材
下山輝昌・松田雄馬・三木孝行 著「Python 実践データ分析100本ノック 第3版」（秀和システム新社、2025）

# 3章 顧客の全体像を把握する１０本ノック

ここでは、スポーツジムの会員データを使って顧客の行動を分析していきます。  
これまでと同様にまずはデータを理解し、加工した後、  
顧客の行動データを分析していきましょう。  
ここでは、機械学習に向けての初期分析を行います。

### ノック21：データを読み込んで把握しよう

In [ ]:
import pandas as pd
uselog = pd.read_csv('use_log.csv')
print(len(uselog))
uselog.head()

#len：データの件数を数える。Excelのcount関数と同じ役割。

In [ ]:
customer = pd.read_csv('customer_master.csv')
print(len(customer))
customer.head()

In [ ]:
class_master = pd.read_csv('class_master.csv')
print(len(class_master))
class_master.head()

In [ ]:
campaign_master = pd.read_csv('campaign_master.csv')
print(len(campaign_master))
campaign_master.head()

### ノック22：顧客データを整形しよう

In [ ]:
customer_join = pd.merge(customer, class_master, on='class', how='left')
customer_join = pd.merge(customer_join, campaign_master, on='campaign_id', how='left')
customer_join.head()

#on：「どの列を基準に結合するか」を指定。
#how：「どのような結合方法を用いるか」を指定。（結合方法↓）
#結合方法'inner'：デフォルト。両方のデータフレームにキーが存在する行のみを残す。
#結合方法'left'：左結合。左側のデータフレームの全ての行を残し、右側からマッチする情報を追加。
#結合方法'right'：右結合。右側のデータフレームの全ての行を残し、左側からマッチする情報を追加。
#結合方法'outer'：完全外部結合。両方のデータフレームのすべての行を結合。マッチしない部分は「NaN」になる。

In [ ]:
print(len(customer))
print(len(customer_join))

In [ ]:
customer_join.isnull().sum()

#isnull().sum()：欠損値（空白のセル）の合計。

### ノック23：顧客データの基礎集計をしよう

In [ ]:
customer_join.groupby('class_name').count()['customer_id']

#groupby：データを同じ仲間ごとにグループ分けする。

In [ ]:
customer_join.groupby('campaign_name').count()['customer_id']

In [ ]:
customer_join.groupby('gender').count()['customer_id']

In [ ]:
customer_join.groupby('is_deleted').count()['customer_id']

In [ ]:
customer_join['start_date'] = pd.to_datetime(customer_join['start_date'])
customer_start = customer_join.loc[customer_join['start_date']>=pd.to_datetime('20180401')]
print(len(customer_start))

#pd.to_datetime：「文字（object）」として読み込まれた日付を、計算や加工ができる『日付専用データ（datetime）』に作り変える命令。
#to：～へ変換する。

#loc[行ラベル,列ラベル]：特定の条件に当てはまるデータだけを書き換える。

### ノック24：最新顧客データの基礎集計をしよう

In [ ]:
customer_join['end_date'] = pd.to_datetime(customer_join['end_date'])
customer_newer = customer_join.loc[(customer_join['end_date']>=pd.to_datetime('20190331'))|(customer_join['end_date'].isna())]
print(len(customer_newer))
customer_newer['end_date'].unique()

#isna() = isnull()　※「欠損値(NaN)セル」があるかチェックする。
#df['列名'].unique：重複を削ぎ落した「純粋な種類の一覧」を1行で表示させる。

In [ ]:
customer_newer.head()

In [ ]:
customer_newer.groupby('class_name')[['customer_id']].count()

In [ ]:
customer_newer.groupby('campaign_name').count()['customer_id']

In [ ]:
customer_newer.groupby('gender').count()['customer_id']

### ノック25：利用履歴データを集計しよう

In [ ]:
uselog.head()

In [ ]:
uselog['usedate'] = pd.to_datetime(uselog['usedate'])
uselog.head()

In [ ]:
uselog['年月'] = uselog['usedate'].dt.strftime('%Y%m')
uselog.head()

#dt.strftime：「dt」datetimeの略。
#strftime：時間を、形式を整えた文字にする。※string = ※strftime = string（文字）format（形式を整える）time（時間）の略。
#決まり文句：「%Y」西暦（4桁）,「%y」西暦（2桁）,「%m」月（2桁）,「%d」日（2桁）,「%H」時（2桁）,「%M」分（2桁）
#例：「%Y」2025,「%y」25,「%m」02,「%d」03,「%H」15,「%M」31

In [ ]:
uselog_months = uselog.groupby(['年月','customer_id'],as_index=False).count()
uselog_months.head()

#groupby：データを同じ仲間ごとにグループ分けする。
#as_index：groupby()の引数。グループ化した項目を、新しい表の「見出し(インデックス)」にするか、「普通のデータ(列)」のままにするか決める。
#as_index = True：グループ名が、新しい表の「見出し(インデックス)」に。
#as_index = False：グループ名がそのまま「普通のデータ(列)」として残る。

In [ ]:
uselog_months.rename(columns={'log_id':'count'}, inplace=True)
uselog_months.head()

#rename：列ラベル(columns)や行ラベル(index)を書き換える。
#columns：「column(列)」の複数形。

#inplace：「『元のデータ』そのものに修正を加える」か、「『コピーした新しいデータ』に修正を加えてユーザーに渡す」か決める。
#inplace = True：『元のデータ』そのものに修正を加える。※ユーザーに渡さない。
#inplace = False：『コピーした新しいデータ』に修正を加えてユーザーに渡す。

In [ ]:
del uselog_months['usedate']
uselog_months.head()

#del：データ(オブジェクト)そのものを抹消するためのPython標準の命令。

#複数の列をまとめて消す：uselog_months.drop(columns=['usedate','usesdate'], inplece=True)
#df.drop()：複数の「列(columns)」や「行(index)」をまとめて削除する。

In [ ]:
uselog_customer = uselog_months.groupby('customer_id', as_index=False).agg(
    mean=('count', 'mean'),
    median=('count', 'median'),
    max=('count', 'max'),
    min=('count','min'))
uselog_customer.head()

#as_index：groupby()の引数。グループ化した項目を、新しい表の「見出し(インデックス)」にするか、「普通のデータ(列)」のままにするか決める。
#as_index = True：グループ名が、新しい表の「見出し(インデックス)」に。
#as_index = False：グループ名がそのまま「普通のデータ(列)」として残る。

#df.agg()：複数の計算（合計、平均、最大値など）をまとめて集計する。

In [ ]:
uselog_customer = uselog_customer.reset_index(drop=False)
uselog_customer.head()

#reset_index：バラバラになったり、行ラベル(index)になってしまった行番号を、上から「0」から順に振り直す。
#reset_index(drop=True)：今の行ラベル(index)を削除して、行番号を新しく振り直す。
#reset_index(drop=False)：今の行ラベル(index)を残しつつ、行番号を新しく振り直す。

### ノック26：利用履歴データから定期利用フラグを作成しよう

In [ ]:
uselog['weekday'] = uselog['usedate'].dt.weekday
uselog.head()

#dt.weekday：指定した日付が「何曜日か」を数字で教えてくれる。曜日は「0～6の数字」で管理される。(↓)
#「0」(月曜日Monday),「1」(火曜日Tuesday),「2」(水曜日Wednesday),「3」(木曜日Thursday),「4」(金曜日Friday),「5」(土曜日Saturday),「6」(日曜日Sunday)

In [ ]:
uselog_weekday = uselog.groupby(['customer_id','年月','weekday'], as_index=False)[['log_id']].count()
uselog_weekday.head()

#as_index：groupby()の引数。グループ化した項目を、新しい表の「見出し(インデックス)」にするか、「普通のデータ(列)」のままにするか決める。
#as_index = True：グループ名が、新しい表の「見出し(インデックス)」に。
#as_index = False：グループ名がそのまま「普通のデータ(列)」として残る。

In [ ]:
uselog_weekday.rename(columns={'log_id':'count'}, inplace=True)
uselog_weekday.head()

#rename：列ラベル(columns)や行ラベル(index)を書き換える。
#columns：「column(列)」の複数形。

#inplace：「『元のデータ』そのものに修正を加える」か、「『コピーした新しいデータ』に修正を加えてユーザーに渡す」か決める。
#inplace = True：『元のデータ』そのものに修正を加える。※ユーザーに渡さない。
#inplace = False：『コピーした新しいデータ』に修正を加えてユーザーに渡す。

In [ ]:
uselog_weekday = uselog_weekday.groupby('customer_id',as_index=False)[['count']].max()
uselog_weekday.head()

In [ ]:
uselog_weekday['routine_fig'] = 0
uselog_weekday.head()

In [ ]:
uselog_weekday['routine_fig'] = uselog_weekday['routine_fig'].where(uselog_weekday['count']<4, 1)
uselog_weekday.head()

#df.where(条件, 書き換え値)：条件に合うデータはそのまま残し、合わないデータは書き換える。
#df.where(uselog_weekday['count']<4, 1)：「count」が「4未満」ならそのまま。そうでなければ、「1」に書き換える。

### ノック27：顧客データと利用履歴データを結合しよう

In [ ]:
uselog_customer.head()

In [ ]:
uselog_weekday.head()

In [ ]:
customer_join.head()

In [ ]:
customer_join = pd.merge(customer_join, uselog_customer, on='customer_id', how='left')
customer_join.head()

#pd.merge()：結合。正規化されてバラバラになったデータを、分析しやすいように「逆正規化（合体）」させる。
#pd.merge(左の表,右の表, left_on=, right_on=, how=)

#left_on：左側の表(テーブル)の使用する列名を指定。
#right_on：右側の表(テーブル)の使用する列名を指定。
#※left_on, right_on はそれぞれの表(テーブル)で使う列名が異なるときに使う。

#how：表(テーブル)の結合方法を指定。（※よく使う結合方法↓）
#how='left'：左の表を主役にする。右に一致するデータがなくても、左のデータはすべて残す。
#how='right'：右の表を主役にする。左に一致するデータがなくても、右のデータはすべて残す。
#how='inner'：両方の表にあるデータだけを残す。どちらか片方にしかないデータは消える。
#how='outer'：両方の表のデータをすべて残す。一致しない部分は「欠損値(NaN)」で埋める。

In [ ]:
customer_join = pd.merge(customer_join, uselog_weekday, on = 'customer_id', how='left')
customer_join.head()

In [ ]:
customer_join.isnull().sum()

### ノック28：会員期間を計算しよう

In [ ]:
from dateutil.relativedelta import relativedelta

#from dateutil.relativedelta：『ライブラリ（dateutil）』に格納される『モジュール（relativedelta）』を指定する。
#import relativedelta：『モジュール（relativedelta）』に格納される『クラス（relativedelta）』を自分のプログラムに導入する。

#ライブラリ（dateutil）：日付や時刻の計算に特化したライブラリ。
#モジュール（relativedelta）：日付計算プログラムを格納したファイル。
#クラス（relativedelta）：日付計算AI。実際に日付計算を行う正体。

#パッケージ：類似性のあるモジュールを格納したフォルダ。
#モジュール：プログラムを再利用するために、コードを格納したファイル。
#クラス：データ型。

In [ ]:
customer_join.head()

In [ ]:
customer_join['calc_date'] = customer_join['end_date']
customer_join['calc_date'] = customer_join['calc_date'].fillna(pd.to_datetime('20190430'))
customer_join.head()

#df.fillna()：「欠損値(NaN)」を()の値で埋める。
#※数値(int,float)だけでなく、文字列(str,object)や日付・時間(datetime)、真偽値(bool値)など、全てのデータ型で使用可能。

In [ ]:
customer_join['membership_period'] = 0
customer_join.head()

In [ ]:
for i in range(len(customer_join)):
    delta = relativedelta(customer_join['calc_date'].iloc[i], customer_join['start_date'].iloc[i])
    customer_join.loc[i, 'membership_period'] = delta.years*12 + delta.months
customer_join.head()

#for i in range：「繰り返し処理」の基本セット
  # for：～の間、繰り返す。
  # i：今、何回目か数えるための変数。
  # in range()：繰り返す回数。

#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[]：整数で指定。整数を使って条件に合うデータを探す。

#delta：変数
#引数「years」：年。クラス（relativedelta）に格納。
#引数「months」：月。クラス（relativedelta）に格納。

### ノック29：顧客行動の各種統計量を把握しよう

In [ ]:
customer_join[['mean', 'median', 'max', 'min']].describe()

#df.describe()：基礎統計量を表示する。

#df.describe()で表示される基礎統計量（↓）
#count(データ数)
#mean(平均値)
#std(標準偏差)
#min(最小値)
#25%(第1四分位数)
#50%(第2四分位数※中央値)
#75%(第3四分位数)
#max(第4四分位数※最大値)

In [ ]:
customer_join.head()

In [ ]:
customer_join.groupby('routine_fig')[['customer_id']].count()

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.hist(customer_join['membership_period'])

#マジックコマンド（% , %%）：プログラムの外側の設定を変更する。
# %（ラインマジック）：その一行だけに効く命令。
# %%（セルマジック）：そのセル全体に効く命令。

#【よく使うマジックコマンド（↓）】
#%matplotlib inline（グラフをノートブック内に直接表示する。）
#%timeit（その行のコードの実行速度を計測する）
#%pwd（現在のフォルダにあるファイル一覧を表示する。）
#%run（外部のPythonファイルをノートブック上で実行する。）

#plt.plot（折れ線グラフ）
#plt.bar（棒グラフ）
#plt.scatter（散布図）
#plt.hist（ヒストグラム）

### ノック30：退会ユーザーと継続ユーザーの違いを把握しよう

In [ ]:
customer_end = customer_join.select_dtypes(include='number').loc[customer_join['is_deleted']==1]
customer_end.describe()

#df.select_dtypes：特定のデータ型の列だけを選んで取り出す。
#join.select_dtypes(include='number')：「数字(number)」を「含める(include)」列だけを選んで取り出す。

#「~」：否定演算子。「NOT（～ではない）」を意味する記号。※flg_is_null：変数。~flg_is_null：flg_is_nullの反対。
#「=」：代入演算子。代入。例）x = 100：xに100を代入。→これ以降、xは100として扱われる。
#「==」：比較演算子。比較。(判定)　例）x == 100：xの中身は100と同じ？→もしxが100ならTrueになる。

In [ ]:
customer_stay = customer_join.select_dtypes(include='number').loc[customer_join['is_deleted']==0]
customer_stay.describe()

In [ ]:
customer_join.to_csv('customer_join.csv', index=False)

#df.to_csv：データをCSV形式で保存する。※分析結果をExcelで確認できるようにする。
#to：～へ変換する。

#index=True：行番号もファイルに書き出す(保存する)。
#index=False：行番号を捨てて、中身だけ保存する。